In [1]:
"""
Q14 – RL Agent Evaluation vs Baseline Policies
================================================
Compares the trained Q-learning agent against four baselines:
  1. Random policy         – uniform random action
  2. Always approve        – action 0 for every claim
  3. Always review         – action 2 for every claim
  4. Supervised model      – CatBoost (best model from Part II); predicts
                             target; maps prediction to action:
                               predicted=1 → Approve (0)
                               predicted=0 → Manual review (2)

Visualisations
--------------
  • Bar chart: total reward per policy
  • Stacked bar: action distribution per policy
  • Heatmap: action choice by true label per policy
  • Confusion-style matrix for RL agent decisions vs true label
  • Q-value distribution (optional)

Dependencies: numpy, pandas, matplotlib, seaborn, scikit-learn
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix


# ── baseline policies ──────────────────────────────────────────────────────────

class RandomPolicy:
    """Choose uniformly at random from 4 actions."""
    name = "Random"

    def __init__(self, n_actions: int = 4, random_state: int = 42):
        self.n_actions = n_actions
        self.rng = np.random.default_rng(random_state)

    def select_action(self, state: np.ndarray) -> int:
        return int(self.rng.integers(0, self.n_actions))


class AlwaysApprovePolicy:
    """Always fast-track (action 0)."""
    name = "Always Approve"

    def select_action(self, state: np.ndarray) -> int:
        return 0


class AlwaysReviewPolicy:
    """Always send to manual review (action 2)."""
    name = "Always Review"

    def select_action(self, state: np.ndarray) -> int:
        return 2


class SupervisedPolicy:
    """
    Supervised model (GradientBoosting) trained on PCA features.
    Maps binary prediction to claim action:
      predicted=1 (benign) → Approve (0)
      predicted=0 (risky)  → Manual review (2)

    Uses GradientBoostingClassifier (no neural networks), which approximates
    the CatBoost approach used in Part II but is available without extra installs.
    """
    name = "Supervised (GBM)"

    def __init__(self, X_train: np.ndarray, y_train: np.ndarray,
                 n_estimators: int = 100, random_state: int = 42):
        print("[Q14] Training supervised GBM policy …")
        self.model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            max_depth=4,
            learning_rate=0.1,
            subsample=0.8,
            random_state=random_state
        )
        self.model.fit(X_train, y_train)
        print("[Q14] Supervised model trained.")

    def select_action(self, state: np.ndarray) -> int:
        pred = self.model.predict(state.reshape(1, -1))[0]
        return 0 if pred == 1 else 2    # approve benign, review risky


# ── evaluation engine ──────────────────────────────────────────────────────────

def evaluate_policy(policy, env) -> dict:
    """
    Run one greedy pass of `policy` over `env`.
    Returns dict with total_reward, action_counts, label_action_matrix.
    """
    state = env.reset()
    total_reward = 0.0
    n_actions    = env.n_actions
    action_counts     = np.zeros(n_actions, dtype=int)
    # label_action[true_label, action]
    label_action      = np.zeros((2, n_actions), dtype=int)
    rewards_per_step  = []
    decisions         = []   # (true_label, action)

    while not env.done:
        action = policy.select_action(state)
        next_state, reward, done, info = env.step(action)

        action_counts[action] += 1
        label_action[info["true_label"], action] += 1
        total_reward += reward
        rewards_per_step.append(reward)
        decisions.append((info["true_label"], action))

        state = next_state if next_state is not None else state

    return {
        "name":             policy.name,
        "total_reward":     total_reward,
        "action_counts":    action_counts,
        "label_action":     label_action,
        "rewards_per_step": rewards_per_step,
        "decisions":        decisions,
        "n_claims":         int(action_counts.sum()),
    }


def run_all_evaluations(agent, train_env, test_env, X_train, y_train):
    """
    Evaluate RL agent and all baselines on the test environment.
    Rebuilds the test env for each policy (reset order is fixed for fairness).

    Parameters
    ----------
    agent      : trained QLearningAgent from q13
    train_env  : training environment (for supervised model fitting)
    test_env   : evaluation environment
    X_train    : raw PCA training features
    y_train    : training labels

    Returns
    -------
    results : list of dicts (one per policy)
    """

    # wrap agent as a policy object
    class AgentPolicy:
        name = "Q-Learning (RL)"
        def __init__(self, ag): self._ag = ag
        def select_action(self, state): return self._ag.select_action(state, greedy=True)

    supervised = SupervisedPolicy(X_train, y_train)

    policies = [
        AgentPolicy(agent),
        RandomPolicy(),
        AlwaysApprovePolicy(),
        AlwaysReviewPolicy(),
        supervised,
    ]

    results = []
    for pol in policies:
        r = evaluate_policy(pol, test_env)
        per_claim = r["total_reward"] / max(r["n_claims"], 1)
        print(f"  {r['name']:<25}  total={r['total_reward']:>9.1f}  per_claim={per_claim:>6.3f}")
        results.append(r)

    return results, supervised


# ── visualisations ─────────────────────────────────────────────────────────────

def plot_comparison(results: list, action_names: list):
    """
    Four-panel comparison plot:
      (a) Total reward per policy
      (b) Action distribution per policy (stacked bar)
      (c) Heatmaps: action by true label for RL agent vs Always Approve
      (d) RL agent decision matrix (true label vs action)
    """
    names      = [r["name"] for r in results]
    totals     = [r["total_reward"] for r in results]
    n_actions  = len(action_names)
    colors_bar = ["#2980b9", "#e74c3c", "#f39c12", "#27ae60", "#8e44ad"]
    action_colors = ["#2ecc71", "#3498db", "#e67e22", "#e74c3c"]

    fig = plt.figure(figsize=(16, 14))
    gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.55, wspace=0.38)

    # ── (0,0) total reward bar ───────────────────────────────────────────────────
    ax0 = fig.add_subplot(gs[0, 0])
    bars = ax0.bar(names, totals, color=colors_bar, edgecolor="white", width=0.6)
    ax0.axhline(0, color="black", linewidth=0.8)
    for bar, val in zip(bars, totals):
        ax0.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(abs(t) for t in totals)*0.01,
                 f"{val:,.0f}", ha="center", va="bottom", fontsize=8, fontweight="bold")
    ax0.set_title("(a) Total Reward per Policy", fontweight="bold")
    ax0.set_ylabel("Total Reward")
    ax0.tick_params(axis="x", rotation=25, labelsize=8)
    ax0.grid(axis="y", alpha=0.3)

    # ── (0,1) per-claim reward ───────────────────────────────────────────────────
    ax1 = fig.add_subplot(gs[0, 1])
    n_claims = [r["n_claims"] for r in results]
    per_claim = [t / n for t, n in zip(totals, n_claims)]
    bars1 = ax1.bar(names, per_claim, color=colors_bar, edgecolor="white", width=0.6)
    ax1.axhline(0, color="black", linewidth=0.8)
    for bar, val in zip(bars1, per_claim):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(abs(p) for p in per_claim)*0.01,
                 f"{val:.3f}", ha="center", va="bottom", fontsize=8, fontweight="bold")
    ax1.set_title("(b) Reward per Claim", fontweight="bold")
    ax1.set_ylabel("Reward / Claim")
    ax1.tick_params(axis="x", rotation=25, labelsize=8)
    ax1.grid(axis="y", alpha=0.3)

    # ── (1, :) stacked action distribution ──────────────────────────────────────
    ax2 = fig.add_subplot(gs[1, :])
    bottoms = np.zeros(len(results))
    for a_idx in range(n_actions):
        vals = [r["action_counts"][a_idx] / r["n_claims"] * 100 for r in results]
        ax2.bar(names, vals, bottom=bottoms, color=action_colors[a_idx],
                label=action_names[a_idx], edgecolor="white")
        # label inside bar if large enough
        for i, (v, b) in enumerate(zip(vals, bottoms)):
            if v > 4:
                ax2.text(i, b + v/2, f"{v:.0f}%", ha="center", va="center",
                         fontsize=7, color="white", fontweight="bold")
        bottoms += vals
    ax2.set_title("(c) Action Distribution per Policy (%)", fontweight="bold")
    ax2.set_ylabel("% of claims")
    ax2.legend(loc="upper right", fontsize=9)
    ax2.tick_params(axis="x", rotation=15, labelsize=8)

    # ── (2,0) RL agent heatmap ───────────────────────────────────────────────────
    ax3 = fig.add_subplot(gs[2, 0])
    rl_result = results[0]
    mat = rl_result["label_action"].astype(float)
    mat_norm = mat / mat.sum(axis=1, keepdims=True) * 100
    sns.heatmap(mat_norm, annot=True, fmt=".1f", cmap="Blues",
                xticklabels=action_names,
                yticklabels=["Risky (0)", "Benign (1)"],
                ax=ax3, cbar_kws={"label": "% within class"})
    ax3.set_title(f"(d) {rl_result['name']}\nAction by True Label (%)", fontweight="bold")
    ax3.set_xlabel("Action chosen"); ax3.set_ylabel("True label")

    # ── (2,1) Supervised model heatmap ──────────────────────────────────────────
    ax4 = fig.add_subplot(gs[2, 1])
    sup_result = results[-1]
    mat2 = sup_result["label_action"].astype(float)
    mat2_norm = mat2 / mat2.sum(axis=1, keepdims=True) * 100
    sns.heatmap(mat2_norm, annot=True, fmt=".1f", cmap="Oranges",
                xticklabels=action_names,
                yticklabels=["Risky (0)", "Benign (1)"],
                ax=ax4, cbar_kws={"label": "% within class"})
    ax4.set_title(f"(e) {sup_result['name']}\nAction by True Label (%)", fontweight="bold")
    ax4.set_xlabel("Action chosen"); ax4.set_ylabel("True label")

    fig.suptitle("Q14 – RL Agent vs Baseline Policies", fontsize=15, fontweight="bold")
    plt.savefig("q14_policy_comparison.png", bbox_inches="tight", dpi=150)
    plt.show()
    print("[Q14] Comparison plot saved to q14_policy_comparison.png")


def plot_rl_decision_patterns(results: list, action_names: list, X_test_pca: np.ndarray):
    """
    Visualise RL agent decisions in PCA space (PC1 vs PC2).
    Each point is a claim coloured by the action the agent took.
    """
    rl_result = results[0]
    decisions  = rl_result["decisions"]   # list of (true_label, action)
    true_labels = np.array([d[0] for d in decisions])
    actions     = np.array([d[1] for d in decisions])

    # X_test_pca rows match env order (not shuffled for test_env)
    X_plot = X_test_pca[:len(decisions)]

    action_colors = ["#2ecc71", "#3498db", "#e67e22", "#e74c3c"]
    markers       = ["o", "^"]   # circle=benign, triangle=risky

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # left: colour by action
    ax = axes[0]
    for a in range(len(action_names)):
        mask = actions == a
        ax.scatter(X_plot[mask, 0], X_plot[mask, 1],
                   c=action_colors[a], label=action_names[a],
                   alpha=0.5, s=10, rasterized=True)
    ax.set_title("RL Agent: Action Chosen (PC1 vs PC2)", fontweight="bold")
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
    ax.legend(markerscale=2, fontsize=9)

    # right: colour by correctness
    ax2 = axes[1]
    # correct = approved benign (0→1) or reviewed risky (2→0)
    correct_approve = (actions == 0) & (true_labels == 1)
    wrong_approve   = (actions == 0) & (true_labels == 0)
    correct_review  = (actions == 2) & (true_labels == 0)
    other           = ~(correct_approve | wrong_approve | correct_review)

    for mask, label, color in [
        (correct_approve, "Correct Approve ✓", "#27ae60"),
        (wrong_approve,   "Risky Approved ✗",  "#e74c3c"),
        (correct_review,  "Correct Review ✓",  "#2980b9"),
        (other,           "Other decisions",    "#bdc3c7"),
    ]:
        ax2.scatter(X_plot[mask, 0], X_plot[mask, 1],
                    c=color, label=label, alpha=0.5, s=10, rasterized=True)
    ax2.set_title("RL Agent: Decision Correctness (PC1 vs PC2)", fontweight="bold")
    ax2.set_xlabel("PC1"); ax2.set_ylabel("PC2")
    ax2.legend(markerscale=2, fontsize=9)

    plt.tight_layout()
    plt.savefig("q14_rl_decision_patterns.png", bbox_inches="tight", dpi=150)
    plt.show()
    print("[Q14] Decision pattern plot saved to q14_rl_decision_patterns.png")


def print_insights(results: list, action_names: list):
    """
    Print a discussion comparing RL vs supervised learning insights.
    """
    print("\n" + "=" * 65)
    print("Q14 – Insights: RL vs Supervised Learning")
    print("=" * 65)

    names   = [r["name"] for r in results]
    totals  = [r["total_reward"] for r in results]
    n_c     = [r["n_claims"] for r in results]
    per_c   = [t / n for t, n in zip(totals, n_c)]

    print("\nPolicy ranking by per-claim reward:")
    ranked = sorted(zip(per_c, names), reverse=True)
    for rank, (pc, name) in enumerate(ranked, 1):
        print(f"  {rank}. {name:<25}  {pc:+.4f} reward/claim")

    rl_idx  = names.index("Q-Learning (RL)")
    sup_idx = names.index("Supervised (GBM)")

    print(f"""
Key Discussion Points
----------------------
1. Reward-aware decision making
   The RL agent optimises directly for the reward function, which penalises
   incorrect approvals of risky claims (-15) much more than unnecessary delays.
   Supervised learning maximises predictive accuracy (AUC) but is blind to the
   asymmetric costs: a supervised model treating FP and FN symmetrically may
   fast-track risky claims more often.

2. Action diversity
   The RL agent has four actions; supervised learning (as implemented) only
   uses Approve/Review. The RL agent can learn to deny or request docs when
   those actions have a higher expected cumulative reward, something supervised
   learning cannot express through a binary prediction.

3. Sequential reasoning
   RL is designed for sequential decisions; here each claim is independent
   (no carry-over state), so the advantage of sequential reasoning is limited.
   In a richer environment (e.g. adjusters tracking open cases), RL would have
   a larger advantage through temporal credit assignment.

4. Exploration cost
   The RL agent incurred poor rewards during training (high ε phase) before
   converging. Supervised learning has no such exploration cost given labelled data.

5. Interpretability
   The tile-coded Q-function is more interpretable than a deep network but
   less so than a logistic regression. The Q-values can be inspected
   per state region, enabling auditors to understand the agent's reasoning.

6. Practical recommendation
   For this dataset (static claims, labelled target available), a supervised
   model trained on a cost-sensitive objective (class_weight, custom loss)
   would likely match RL performance with less complexity. RL shines when
   labels are absent or the environment is truly sequential/dynamic.
""")
    print("=" * 65)


# ── full pipeline ──────────────────────────────────────────────────────────────

def run_q14(agent, train_env, test_env, X_test_pca: np.ndarray, y_test: np.ndarray):
    """
    Entry point for Q14. Requires trained agent from q13.

    Parameters
    ----------
    agent       : QLearningAgent from q13_rl_training.train_agent()
    train_env   : ClaimsProcessingEnv (training split)
    test_env    : ClaimsProcessingEnv (test split)
    X_test_pca  : np.ndarray, PCA features of test split
    y_test      : np.ndarray, true labels for test split
    """
    print("\n[Q14] Evaluating all policies …\n")
    results, supervised = run_all_evaluations(
        agent, train_env, test_env,
        X_train=train_env.X, y_train=train_env.y
    )

    action_names = test_env.action_names

    plot_comparison(results, action_names)
    plot_rl_decision_patterns(results, action_names, X_test_pca)
    print_insights(results, action_names)

    return results


# ── main ────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    import sys, os
    sys.path.insert(0, os.path.dirname(__file__))
    from q12_rl_environment import build_claims_env
    from q13_rl_training import train_agent

    CSV_PATH = "test_data.csv"   # <- update path as needed

    train_env, test_env, pca, scaler, X_te, y_te = build_claims_env(CSV_PATH)

    agent, eval_rewards, eval_episodes = train_agent(
        train_env, test_env, n_episodes=60, eval_every=5
    )

    run_q14(agent, train_env, test_env, X_te, y_te)

NameError: name '__file__' is not defined

In [2]:
if __name__ == "__main__":
    import sys
    import os

    # os.getcwd() works perfectly in Jupyter Notebooks
    sys.path.insert(0, os.getcwd())

    from q12_rl_environment import build_claims_env
    from q13_rl_training import train_agent

    CSV_PATH = "test_data.csv"  # <- update path as needed

    train_env, test_env, pca, scaler, X_te, y_te = build_claims_env(CSV_PATH)

    agent, eval_rewards, eval_episodes = train_agent(
        train_env, test_env, n_episodes=60, eval_every=5
    )

    # Note: Make sure run_q14 is defined somewhere above this cell!
    run_q14(agent, train_env, test_env, X_te, y_te)

ModuleNotFoundError: No module named 'q12_rl_environment'